# Creating Machine Learning Ready Datasets

**Important Note**: This notebook was updated on 03.18.26 in order to address the order in which we created the dataset. Prior to this change, we forward filled all data which included weekends and holidays. After the update, we now only forward fill the MacroEconomic data and then use the price data to drop non-trading days. Check version histories if desired.

The purpose of this notebook is to merge the macroeconomic dataset and the price data. We need to forward fill the data that comes infrequently. We then need to calculate the forward looking log retruns (by one quarter) and annualize it, and then calculate the forward looking volatility and annualize that. These vectors will be our labels.

After that, I want to create a second CSV that uses stationary data (differenced) instead of the raw data to see whether or not this imporves the performance of our LSTM. This dataset will be more similar to how financial analysts and economoists look at data, however, it is possible that deep learning will find similar patterns in the raw data because of the nature of deep learning. We will compare model performance with the two datasets.

## Import Libraries
Let's create a section for all of the libraries that will be necessary for this manipulation.

In [22]:
# Manipulation libraries  
import numpy as np
import pandas as pd
import altair as alt

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

# Import the datasets
Now, we can pull in both of the CSV files that we will merge together.

In [23]:
macro_df = pd.read_csv('fred_data.csv')
price_df = pd.read_csv('yfinance_data.csv')
print(f"Macro DataFrame shape: {macro_df.shape}, Price DataFrame shape: {price_df.shape}")

Macro DataFrame shape: (20141, 26), Price DataFrame shape: (24955, 38)


Now, let's take a look at the dataframes to make sure we can merge on the 'date'.

In [24]:
macro_df.head()

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,female_unemployment_rate,average_duration_of_unemployment,one_month_yield,three_month_yield,six_month_yield,one_year_yield,two_year_yield,five_year_yield,ten_year_yield,thirty_year_yield
0,1946-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1946-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1946-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1946-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947-01-01,243.164,2182.681,NaN,5.352,21.48,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
price_df.head()

,date,BIL,BND,GLD,HYG_x,IEF,IWM,LQD_x,QQQ,SPY,...,NG=F,SHY,TLT_y,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


This is the first large adaptation after running our first model where we artificially used data from weekends and holidays. Let's forward fill the macro data before merging.

In [26]:
macro_filled_df = macro_df.ffill()
macro_filled_df.head(10)

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,female_unemployment_rate,average_duration_of_unemployment,one_month_yield,three_month_yield,six_month_yield,one_year_yield,two_year_yield,five_year_yield,ten_year_yield,thirty_year_yield
0,1946-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1946-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1946-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1946-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947-01-01,243.164,2182.681,NaN,5.352,21.48,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1947-02-01,243.164,2182.681,NaN,5.352,21.62,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1947-03-01,243.164,2182.681,NaN,5.352,22.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1947-04-01,245.968,2176.892,NaN,5.360,22.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1947-05-01,245.968,2176.892,NaN,5.360,21.95,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1947-06-01,245.968,2176.892,NaN,5.360,22.08,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Great, so this should be a simple horizontal merge of the dataframes on the 'date' column.

In [27]:
merged_df = pd.merge(macro_filled_df, price_df, on='date', how='right')
merged_df.shape

(24955, 63)

Let's do a quick sanity check to see the minimum and maximum date.

In [28]:
min, max = merged_df['date'].min(), merged_df['date'].max()
print(f"Date range: {min} to {max}")

Date range: 1927-12-30 to 2026-03-18


Alright, this looks great. Let's pull all of the column names into a list so we can work with them a little easier.

In [29]:
independent_columns = merged_df.columns.tolist()
print(f"Columns in merged DataFrame: {independent_columns}")

Columns in merged DataFrame: ['date', 'nominal_GDP', 'real_GDP', 'debt_to_GDP', 'debt_interest', 'consumer_price_index', 'core_PCI', 'personal_consumption_expenditure', 'core_PCE', 'producer_price_index', 'unemployment_rate', 'initial_jobless_claims', 'continued_jobless_claims', 'teenager_unemployment_rate', 'adult_unemployment_rate', 'male_unemployment_rate', 'female_unemployment_rate', 'average_duration_of_unemployment', 'one_month_yield', 'three_month_yield', 'six_month_yield', 'one_year_yield', 'two_year_yield', 'five_year_yield', 'ten_year_yield', 'thirty_year_yield', 'BIL', 'BND', 'GLD', 'HYG_x', 'IEF', 'IWM', 'LQD_x', 'QQQ', 'SPY', 'TIP', 'TLT_x', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY', 'CL=F', 'DX-Y.NYB', 'GC=F', 'HG=F', 'HYG_y', 'LQD_y', 'NG=F', 'SHY', 'TLT_y', '^FTSE', '^GSPC', '^HSI', '^MOVE', '^TNX', '^VIX', '^VXN']


Interestingly, we can see that we had some duplicate data (i.e. TLT) when we merged our data in the previous CSVs. So, let's loop through and keep only one of them.

In [30]:
# Drop _y duplicates and rename _x back to original
x_cols = [col for col in merged_df.columns if col.endswith('_x')]

for col in x_cols:
    base = col[:-2]  # strip '_x'
    merged_df.drop(columns=[f'{base}_y'], inplace=True)
    merged_df.rename(columns={col: base}, inplace=True)

Now, let's forward fill the data so that less frequent data is filled into the dataframe so we have complete vectors at each timeframe. First, let's get an idea of the number of NaN's in each column, then forward fill, then double check to see if it is filling what we expect (like GDP)

In [31]:
 merged_df.isna().sum()

date                                    0
nominal_GDP                          8396
real_GDP                             8396
debt_to_GDP                          9519
debt_interest                        8396
consumer_price_index                 8396
core_PCI                             8473
personal_consumption_expenditure     8487
core_PCE                             8487
producer_price_index                20731
unemployment_rate                    8402
initial_jobless_claims               9775
continued_jobless_claims             9775
teenager_unemployment_rate           8402
adult_unemployment_rate              8402
male_unemployment_rate               8402
female_unemployment_rate             8402
average_duration_of_unemployment     8402
one_month_yield                     18597
three_month_yield                   13473
six_month_yield                     13473
one_year_yield                       8511
two_year_yield                      12146
five_year_yield                   

In [32]:
filled_df = merged_df.copy()

Alright, this looks pretty good. The first 63 columns are our independent variables at the moment. Let's start building the log returns labels. We will need log returns of all of the potential ETF's that we will be using in our portfolio generator. So let's create those.

In [33]:
target_variables = ['XLF','XLK','XLU','XLV','XLE','XLI','XLB','XLP','XLY','XLRE','BIL','IEF','TLT','LQD','HYG','TIP','GLD']

Now that we have our target variables, let's iterate through and create the new columns for our target labels. I am going to append '_target' on the end so that we remember to never use these as features to avoid data leakage.

The first thing we are doing is calculating the log return that is a quarter forward looking (63 days assuming 252 trading days). We will then multiply this by 4 to annualize it.

In [34]:
for ticker in target_variables:
    filled_df[f'{ticker}_logreturn_target'] = np.log(filled_df[ticker].shift(-63)/filled_df[ticker]) * 4

log_return_columns = [f'{ticker}_logreturn_target' for ticker in target_variables]
log_return_columns

['XLF_logreturn_target',
 'XLK_logreturn_target',
 'XLU_logreturn_target',
 'XLV_logreturn_target',
 'XLE_logreturn_target',
 'XLI_logreturn_target',
 'XLB_logreturn_target',
 'XLP_logreturn_target',
 'XLY_logreturn_target',
 'XLRE_logreturn_target',
 'BIL_logreturn_target',
 'IEF_logreturn_target',
 'TLT_logreturn_target',
 'LQD_logreturn_target',
 'HYG_logreturn_target',
 'TIP_logreturn_target',
 'GLD_logreturn_target']

Alright perfect. Now, let's build all of the annualized volatility columns.

In [35]:
for ticker in target_variables:
    filled_df[f'{ticker}_volatility_target'] = (
        np.log(filled_df[ticker] / filled_df[ticker].shift(1))
          .rolling(63)
          .std()
        * np.sqrt(252)
    ).shift(-63)
    # Final .shift(-63) required to align label with date of prediction

volatility_columns = [f'{ticker}_volatility_target' for ticker in target_variables]
volatility_columns

['XLF_volatility_target',
 'XLK_volatility_target',
 'XLU_volatility_target',
 'XLV_volatility_target',
 'XLE_volatility_target',
 'XLI_volatility_target',
 'XLB_volatility_target',
 'XLP_volatility_target',
 'XLY_volatility_target',
 'XLRE_volatility_target',
 'BIL_volatility_target',
 'IEF_volatility_target',
 'TLT_volatility_target',
 'LQD_volatility_target',
 'HYG_volatility_target',
 'TIP_volatility_target',
 'GLD_volatility_target']

Alright, let's export this as a CSV that can now be worked on. It's important to note that this still has a significant number of NaN values and independent variables that we may end up dropping in our modelling but it is a workable dataset to start designing our machine learning pipeline around.

In [36]:
#Let's confirm we don't have any weekends orholidays
filled_df.tail(20)

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
24935,2026-02-19,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24936,2026-02-20,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24937,2026-02-23,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24938,2026-02-24,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24939,2026-02-25,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24940,2026-02-26,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24941,2026-02-27,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24942,2026-03-02,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24943,2026-03-03,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24944,2026-03-04,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
filled_df.drop(columns=['DX-Y.NYB'], inplace=True)
filled_df.to_csv('raw_data_prediction_dataset.csv', index=False)

Let's take a quick look visually to see if these targets make sense.

In [38]:
XLK_return_df = filled_df[['date', 'XLU_logreturn_target', 'XLK_logreturn_target']].copy().dropna()

# Mean lines
xlk_mean = XLK_return_df['XLK_logreturn_target'].mean()
xlu_mean = XLK_return_df['XLU_logreturn_target'].mean()

mean_df = pd.DataFrame({
    'date': [XLK_return_df['date'].min(), XLK_return_df['date'].max()],
    'xlk_mean': xlk_mean,
    'xlu_mean': xlu_mean
})

XLK_return = alt.Chart(XLK_return_df).mark_line().encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_logreturn_target:Q', axis=alt.Axis(title='Log Return', grid=False))
)
XLU_return = alt.Chart(XLK_return_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_logreturn_target:Q'
)

xlk_mean_line = alt.Chart(mean_df).mark_line(strokeDash=[6,3]).encode(
    x='date:T',
    y='xlk_mean:Q'
)
xlu_mean_line = alt.Chart(mean_df).mark_line(strokeDash=[6,3], color='orange').encode(
    x='date:T',
    y='xlu_mean:Q'
)

(XLK_return + XLU_return + xlk_mean_line + xlu_mean_line).properties(
    title='XLK vs XLU Log Returns', height=400, width=900).configure_view(strokeWidth=0)

alt.LayerChart(...)

In [39]:
XLK_volatility_df = filled_df[['date', 'XLU_volatility_target', 'XLK_volatility_target']].copy().dropna()

xlk_vol_mean = XLK_volatility_df['XLK_volatility_target'].mean()
xlu_vol_mean = XLK_volatility_df['XLU_volatility_target'].mean()

vol_mean_df = pd.DataFrame({
    'date': [XLK_volatility_df['date'].min(), XLK_volatility_df['date'].max()],
    'xlk_mean': xlk_vol_mean,
    'xlu_mean': xlu_vol_mean
})

XLK_volatility = alt.Chart(XLK_volatility_df).mark_line().encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_volatility_target:Q', axis=alt.Axis(title='Volatility', grid=False))
)
XLU_volatility = alt.Chart(XLK_volatility_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_volatility_target:Q'
)

xlk_vol_mean_line = alt.Chart(vol_mean_df).mark_line(strokeDash=[6,3]).encode(
    x='date:T',
    y='xlk_mean:Q'
)
xlu_vol_mean_line = alt.Chart(vol_mean_df).mark_line(strokeDash=[6,3], color='orange').encode(
    x='date:T',
    y='xlu_mean:Q'
)

(XLK_volatility + XLU_volatility + xlk_vol_mean_line + xlu_vol_mean_line).properties(
    title='XLK vs XLU Volatilities', height=400, width=900).configure_view(strokeWidth=0)

alt.LayerChart(...)

In [40]:
XLK_complete_df = filled_df[['date', 'XLK_logreturn_target', 'XLK_volatility_target']].copy().dropna()

base = alt.Chart(XLK_complete_df).encode(x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)))

XLK_volatility = base.mark_line().encode(
    y=alt.Y('XLK_volatility_target:Q', axis=alt.Axis(title='Volatility', grid=False, titleColor='steelblue', labelColor='steelblue'))
)

XLK_return = base.mark_line(color='red').encode(
    y=alt.Y('XLK_logreturn_target:Q', axis=alt.Axis(title='Log Return', grid=False, titleColor='red', labelColor='red'))
)

alt.layer(XLK_return,XLK_volatility).resolve_scale(
    y='independent'
).properties(
    title='XLK Log Returns and Volatility', height=400, width=900
).configure_view(strokeWidth=0)

alt.LayerChart(...)